In [9]:
import pandas as pd

In [10]:
rawData = pd.read_excel('./data/retail_data_synthetic_50k.xlsx')

In [11]:
print(rawData.head().to_markdown())

|    | Transaction_ID   | Customer_ID   | Gender     |   Age | Category    |   Quantity |   Unit_Price |   Discount | Date                | Store_Region   | Online_Or_Offline   | Payment_Method   |   Total_Amount |
|---:|:-----------------|:--------------|:-----------|------:|:------------|-----------:|-------------:|-----------:|:--------------------|:---------------|:--------------------|:-----------------|---------------:|
|  0 | TXN-00000        | CUST-1127     | Female     |    60 | Furniture   |          9 |       202.12 |       0.26 | 2023-02-12 00:00:00 | North          | Online              | Digital Wallet   |        1346.12 |
|  1 | TXN-00001        | CUST-1460     | Male       |    30 | Beauty      |          4 |        75.03 |       0.05 | 2021-12-04 00:00:00 | West           | Online              | Digital Wallet   |         285.11 |
|  2 | TXN-00002        | CUST-0861     | Male       |    52 | Clothing    |          1 |       374.66 |       0.15 | 2020-10-21 00:00:00 | 

# Data Cleaning

In [12]:
rawData.info()
rawData.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Transaction_ID     50000 non-null  object        
 1   Customer_ID        50000 non-null  object        
 2   Gender             50000 non-null  object        
 3   Age                50000 non-null  int64         
 4   Category           50000 non-null  object        
 5   Quantity           50000 non-null  int64         
 6   Unit_Price         50000 non-null  float64       
 7   Discount           50000 non-null  float64       
 8   Date               50000 non-null  datetime64[ns]
 9   Store_Region       50000 non-null  object        
 10  Online_Or_Offline  50000 non-null  object        
 11  Payment_Method     50000 non-null  object        
 12  Total_Amount       50000 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(2), object(7)
memory 

,Age,Quantity,Unit_Price,Discount,Date,Total_Amount
count,50000.000000,50000.000000,50000.000000,50000.000000,50000,50000.000000
mean,43.425100,4.986980,252.122316,0.200063,2021-12-28 15:02:42.432000256,1004.289097
min,18.000000,1.000000,5.010000,0.000000,2020-01-01 00:00:00,3.480000
25%,30.000000,3.000000,127.837500,0.100000,2020-12-27 00:00:00,321.007500
50%,43.000000,5.000000,252.350000,0.200000,2021-12-28 00:00:00,765.855000
75%,56.000000,7.000000,375.662500,0.300000,2022-12-31 00:00:00,1488.305000
max,69.000000,9.000000,499.990000,0.400000,2023-12-31 00:00:00,4477.140000
std,15.009896,2.577119,143.094619,0.115804,NaN,845.558082


In [13]:
cleanData = rawData.copy()
cleanData.columns = [col.lower() for col in cleanData.columns]


In [14]:
mathErrors = cleanData[cleanData['total_amount'] != round(((cleanData['unit_price'] * cleanData['quantity']) - (cleanData['unit_price'] * cleanData['discount'] * cleanData['quantity'])),2)]
print(len(mathErrors))
print(mathErrors.head().to_markdown())

114
|      | transaction_id   | customer_id   | gender   |   age | category   |   quantity |   unit_price |   discount | date                | store_region   | online_or_offline   | payment_method   |   total_amount |
|-----:|:-----------------|:--------------|:---------|------:|:-----------|-----------:|-------------:|-----------:|:--------------------|:---------------|:--------------------|:-----------------|---------------:|
|  720 | TXN-00720        | CUST-0397     | Male     |    63 | Furniture  |          1 |       267.5  |       0.39 | 2021-01-30 00:00:00 | East           | In-store            | Cash             |         163.17 |
| 1121 | TXN-01121        | CUST-0542     | Male     |    48 | Grocery    |          5 |        75.77 |       0.3  | 2021-10-22 00:00:00 | South          | In-store            | Digital Wallet   |         265.19 |
| 1354 | TXN-01354        | CUST-1811     | Male     |    35 | Beauty     |          6 |       453.75 |       0.07 | 2022-11-05 00:00:00 | W

In [15]:
def checkTotal(row):
    rawTotal = row['quantity'] * row['unit_price']
    discount = rawTotal * row['discount']
    calculatedTotal = round(rawTotal - discount,2)
    difference = abs(row['total_amount'] - calculatedTotal)
    row['has_error'] = difference > .02
    return row

mathErrors = mathErrors.apply(checkTotal, axis=1)
print(f'number of records with arithmitic errors: {len(mathErrors[mathErrors['has_error']])}')

number of records with arithmitic errors: 0


In [16]:
# Drop Transaction_ID and Customer_ID columns
cleanData = cleanData.drop(columns=['transaction_id', 'customer_id'])

In [17]:
cleanData.to_csv('./data/clean_data.csv', index=False)